# Global Income Inequality — reproducible walkthrough

This notebook is a thin interface to the tested `global_inequality` package. It contains no duplicated Gini, Lorenz, data-preparation, simulation, or plotting implementation. Run `uv sync --locked --group dev` before opening it, or use the command-line reproduction path documented in `README.md`.

> The subsidy and progressive-tax examples are **illustrative static simulations, not policy forecasts**. See `docs/methodology.md` for assumptions and limitations.

In [ ]:
from pathlib import Path

import pandas as pd

from global_inequality.data import KEY_COLUMNS, load_percentiles, prepare_percentiles
from global_inequality.lorenz import gini_from_group
from global_inequality.report import build_report
from global_inequality.simulation import simulate_bottom_subsidy, simulate_progressive_tax

## Load and validate the committed sample

Preparation selects national observations, requires complete 100-bin distributions, and fails on missing analytical values instead of dropping or imputing bins.

In [ ]:
sample_path = Path("data/sample/pip_percentiles_sample.csv")
data = prepare_percentiles(load_percentiles(sample_path), reporting_level="national")
data.shape

## Recompute grouped-data Gini coefficients

Welfare type and survey year remain part of every result key; income and consumption series are not silently treated as identical indicators.

In [ ]:
gini = (
    data.groupby(KEY_COLUMNS, sort=True)
    .apply(gini_from_group, include_groups=False)
    .rename("gini")
    .reset_index()
)
gini

## Illustrative simulations — not policy forecasts

The first scenario assumes an externally funded fixed addition to the bottom 50%. The second is a static, arithmetically budget-neutral top-10 tax / bottom-40 transfer. Neither models behavior, administration, incidence, equilibrium effects, avoidance, or uncertainty.

In [ ]:
indonesia_2019 = data.loc[(data["country_code"] == "IDN") & (data["year"] == 2019)]
subsidy = simulate_bottom_subsidy(indonesia_2019)
progressive_tax = simulate_progressive_tax(indonesia_2019)
pd.DataFrame(
    [
        {"scenario": "Observed", "gini": gini_from_group(indonesia_2019)},
        {"scenario": subsidy.name, "gini": subsidy.gini},
        {"scenario": progressive_tax.name, "gini": progressive_tax.gini},
    ]
)

## Generate all static visuals and the report

The report builder owns visual generation so notebook and command-line output cannot drift.

In [ ]:
build_report(sample_path, Path("report"))